In [0]:
PATH_ORDER_FILES = "/Volumes/customer_360/raw/source_files/landing_data/orders/"
TABLE_BRONZE_ORDER = "customer_360.bronze.orders"
TABLE_METRIC = "customer_360.raw.stream_metrics"
PATH_ORDER_CHECKPOINTLOCATION_BRONZE = "/Volumes/customer_360/raw/source_files/checkpoints/orders/"

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {TABLE_BRONZE_ORDER} (
        order_id STRING NOT NULL,
        customer_id STRING NOT NULL,
        product_id STRING NOT NULL,
        order_status STRING,
        quantity INT,
        total_amount DECIMAL(12,2),
        order_date TIMESTAMP,
        updated_at TIMESTAMP
        )
        USING DELTA;
          """)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)

order_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("total_amount", DecimalType(12, 2), True),
    StructField("order_date", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
orders_bronze=(
    spark
    .readStream
    .format("csv")
    .option("header",True)
    .schema(order_schema)
    .load(PATH_ORDER_FILES)
)

In [0]:
query=(
    orders_bronze
    .writeStream
    .trigger(availableNow=True)
    .format("delta")
    .option("checkpointlocation",PATH_ORDER_CHECKPOINTLOCATION_BRONZE)
    .outputMode("append")
    .toTable(TABLE_BRONZE_ORDER)
)
query.awaitTermination()

In [0]:

import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="order_bronze",
            batch_id=int(progress["batchId"]),
            input_rows=int(source.get("numInputRows", 0)),
            input_rows_per_second=float(source.get("inputRowsPerSecond", 0.0)),
            processed_rows_per_second=float(source.get("processedRowsPerSecond", 0.0)),
            processing_time_ms=int(
                progress.get("durationMs", {}).get("triggerExecution", 0)
            )
        )
    )

if metrics:
    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)
        

In [0]:
display(
    spark.sql(f"select * from {TABLE_BRONZE_ORDER}")

)